# Modelo A2 — Vecinos por perfil + popularidad

## Hipótesis

La similitud entre perfiles aporta información sobre las preferencias del usuario, pero los resultados del modelo anterior muestran que esta señal por sí sola no logra superar al baseline.

En esta versión se mantiene la ponderación de los vecinos según su similitud y se incorpora además la popularidad de los productos dentro del conjunto de entrenamiento.

El objetivo es combinar una señal personalizada, basada en el perfil del usuario, con una señal general, basada en la frecuencia de interacciones positivas de cada producto.

## Método

El modelo sigue los siguientes pasos:

1. Codifica las cuatro características del usuario:
   - tipo de piel
   - tono de piel
   - color de ojos
   - color de cabello

2. Calcula la similitud coseno entre el perfil del usuario y los perfiles disponibles en entrenamiento.

3. Selecciona los K usuarios más similares.

4. Busca los productos con interacciones positivas de esos vecinos. Cada producto recibe un puntaje según la similitud del vecino que lo recomendó.

5. Calcula además la popularidad de los productos a partir de las interacciones positivas del conjunto de entrenamiento.

6. Normaliza ambos puntajes y los combina mediante el parámetro `alpha`.

7. Ordena los productos según el puntaje final y genera el Top-10.

`K` representa la cantidad de vecinos considerados y `alpha` determina cuánto peso tiene la información del perfil frente a la popularidad.

## Ejemplo conceptual

Supongamos un usuario con piel grasa, tono oscuro, ojos azules y cabello castaño.

El modelo primero busca usuarios con perfiles similares y analiza qué productos tuvieron interacciones positivas entre esos vecinos.

A diferencia de la versión anterior, también tiene en cuenta la popularidad general de cada producto dentro del conjunto de entrenamiento.

Por ejemplo, si `alpha = 0.75`, el puntaje final combina:

- 75 % de la señal obtenida a partir de usuarios con perfiles similares.
- 25 % de la popularidad del producto.

De esta manera se busca mantener la personalización del modelo anterior, pero incorporando una señal general que pueda mejorar el ranking de las recomendaciones.


## 1. Librerías, datos y división de los datos

En esta primera parte se importan las librerías necesarias y se carga `sephora_common.py`, que contiene las funciones compartidas para la preparación, división y evaluación de los datos.

Para mantener la misma comparación entre los modelos, se utilizan:

- 500 usuarios para validación.
- 500 usuarios para test.
- Semilla aleatoria = 42.

Los datos utilizados provienen de `data/processed/`, principalmente de `reviews_clean.parquet` y `products_clean.parquet`.

In [2]:
from pathlib import Path
import sys
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from IPython.display import display


# Buscamos la carpeta principal del proyecto
PROJECT_ROOT = next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se encontró la carpeta data/processed"
    )


# Buscamos el archivo con las funciones comunes
helper_dir = next(
    (
        p for p in [Path.cwd(), PROJECT_ROOT / "notebooks"]
        if (p / "sephora_common.py").exists()
    ),
    None
)

if helper_dir is None:
    raise FileNotFoundError(
        "No se encontró sephora_common.py"
    )


# Agregamos la carpeta para poder importar el archivo
sys.path.insert(0, str(helper_dir))

import sephora_common as sc
from sephora_common import PROFILE_COLS, rank_scores


# Cantidad de usuarios que usamos para evaluar
N_VALIDACION = 500
N_TEST = 500

# Semilla para obtener siempre la misma división de datos
SEMILLA = 42


# Cargamos los datos y dejamos preparado el contexto del experimento
ctx = sc.load_context(
    N_VALIDACION,
    N_TEST,
    SEMILLA
)


print("Protocolo:", ctx.protocol_id[:12])
print("Resultados:", ctx.output_dir)


# Mostramos la información general del experimento
datos_manifest = {
    k: v
    for k, v in ctx.manifest.items()
    if k not in ["validation_cases", "test_cases", "versions"]
}

display(
    pd.Series(datos_manifest).to_frame("valor")
)

display(
    pd.DataFrame(
        ctx.manifest["versions"],
        index=[0]
    )
)


Protocolo: 99fc64b0ea63
Resultados: c:\Users\Administrator\Documents\DATA SCIENCE\PF\sephora-recommender\artifacts\comparacion_4_modelos\99fc64b0ea63


,valor
version,cold-start-users-v2.0
seed,42
protocol_id,99fc64b0ea63f0e1e20f6a8a2d96aee1222e401c635a55...
reviews_input,1093855
reviews_valid_before_pair_dedup,1093855
reviews_unique_pairs,1088886
removed_duplicate_pairs,4969
catalog_skincare,2420
users_total,503216
eligible_users,161396


,python,numpy,pandas,scipy,scikit_learn
0,3.14.5,2.4.6,3.0.3,1.18.1,1.9.0


## 2. Condiciones comunes de comparación

Para poder comparar los modelos de forma justa, todos utilizan las mismas condiciones de evaluación.

- **Datos del usuario:** tipo de piel, tono de piel, color de ojos y color de cabello. Cada modelo puede utilizar estas características de manera diferente.
- **Productos:** se trabaja únicamente con productos de la categoría Skincare.
- **Interacción positiva:** se considera positiva cuando el usuario dio un `rating >= 4` o cuando `is_recommended = True`.
- **Validación:** se utilizan 500 usuarios para probar las distintas configuraciones y elegir la mejor.
- **Test:** se utilizan otros 500 usuarios para evaluar la configuración seleccionada.
- **Producto objetivo:** para cada usuario se oculta un producto positivo y se evalúa si el modelo logra recomendarlo dentro del Top-10.
- **Entrenamiento:** los usuarios utilizados para validación y test no participan del conjunto de entrenamiento.

Para la evaluación se seleccionan usuarios que tengan al menos dos interacciones positivas y que cuenten con información de tipo y tono de piel.

Los productos positivos que se utilizan para evaluar al usuario permanecen ocultos para el modelo. De esta manera se busca evitar que el modelo utilice información que no debería conocer al momento de generar la recomendación.

La misma división de datos y la misma semilla se mantienen para todos los modelos, permitiendo comparar sus resultados bajo las mismas condiciones.

## 3. Cómo funciona el algoritmo

El modelo combina dos señales para generar las recomendaciones:

1. **Preparación de los perfiles:** se toman los usuarios del conjunto de entrenamiento y sus cuatro características: tipo de piel, tono de piel, color de ojos y color de cabello. Estas variables se transforman para poder comparar los perfiles.

2. **Búsqueda de vecinos:** para cada usuario se calcula la similitud coseno con los perfiles de entrenamiento y se seleccionan los K usuarios más parecidos.

3. **Puntaje por similitud:** se buscan los productos con interacciones positivas de esos vecinos. Cada producto recibe un puntaje según la similitud del vecino que lo recomendó.

4. **Puntaje por popularidad:** se calcula la popularidad de cada producto utilizando las interacciones positivas del conjunto de entrenamiento.

5. **Combinación de los puntajes:** los puntajes de perfil y popularidad se normalizan y se combinan utilizando el parámetro `alpha`.

   - Un `alpha` alto da mayor importancia a la similitud de perfil.
   - Un `alpha` bajo da mayor importancia a la popularidad.

6. **Generación del Top-10:** los productos se ordenan según el puntaje final y se seleccionan los 10 primeros.

### Limitación

El modelo continúa dependiendo de solo cuatro características del usuario para representar sus preferencias. Además, al incorporar popularidad, algunos productos muy frecuentes pueden ganar posiciones en el ranking aunque no sean los más relacionados con el perfil particular del usuario.

In [3]:
class ProfilePopularity:
    """
    Recomendador que combina dos señales:

    1. Productos valorados positivamente por usuarios con perfiles similares.
    2. Popularidad de los productos dentro del conjunto de entrenamiento.
    """

    def __init__(self, ctx):

        # Usuarios que pertenecen al conjunto de entrenamiento
        self.profiles = ctx.profiles.loc[
            sorted(ctx.train.author_id.unique())
        ]

        # Convertimos las características del perfil a variables numéricas
        self.encoder = DictVectorizer(sparse=True)

        records = self.profiles.to_dict("records")

        self.encoder.fit(
            records + [{c: "Unknown" for c in PROFILE_COLS}]
        )


        # Valores permitidos para cada característica
        self.allowed = {
            c: set(self.profiles[c]) | {"Unknown"}
            for c in PROFILE_COLS
        }


        # Agrupamos usuarios que tienen exactamente el mismo perfil
        self.groups = {}

        for uid, row in self.profiles.iterrows():

            perfil = tuple(
                row[c] for c in PROFILE_COLS
            )

            self.groups.setdefault(
                perfil,
                []
            ).append(uid)


        # Guardamos los perfiles distintos
        self.signatures = list(
            self.groups
        )


        # Transformamos los perfiles para calcular similitud
        self.X = self.encoder.transform([
            dict(zip(PROFILE_COLS, perfil))
            for perfil in self.signatures
        ])


        # Guardamos los productos con interacción positiva de cada usuario
        self.positives = (
            ctx.train[ctx.train.positive]
            .groupby("author_id")
            .product_id
            .apply(list)
            .to_dict()
        )


        # Calculamos la popularidad de cada producto
        # utilizando solamente los datos de entrenamiento
        popularidad = (
            ctx.train[ctx.train.positive]
            .groupby("product_id")
            .size()
            .reset_index(name="popularity")
        )


        # Normalizamos la popularidad entre 0 y 1
        max_popularity = popularidad["popularity"].max()

        popularidad["popularity_score"] = (
            popularidad["popularity"] / max_popularity
        )


        # Guardamos el puntaje de popularidad de cada producto
        self.popularity = dict(
            zip(
                popularidad["product_id"],
                popularidad["popularity_score"]
            )
        )


        self.n_train_users = len(
            self.profiles
        )


    def neighbors(self, profile, k):

        if not 1 <= k <= self.n_train_users:
            raise ValueError(
                f"K={k} inválido para "
                f"{self.n_train_users} usuarios de entrenamiento"
            )


        # Armamos el perfil del usuario
        key = tuple(
            profile.get(c, "Unknown")
            if profile.get(c, "Unknown") in self.allowed[c]
            else "Unknown"
            for c in PROFILE_COLS
        )


        # Transformamos el perfil
        query = self.encoder.transform([
            dict(zip(PROFILE_COLS, key))
        ])


        # Calculamos la similitud coseno
        similarities = cosine_similarity(
            query,
            self.X
        ).ravel()


        vecinos = []


        # Ordenamos los perfiles desde el más parecido
        orden = np.argsort(
            -similarities
        )


        for indice in orden:

            similitud = similarities[indice]

            usuarios_grupo = self.groups[
                self.signatures[indice]
            ]


            for uid in usuarios_grupo:

                vecinos.append(
                    (uid, similitud)
                )

                if len(vecinos) == k:
                    return vecinos


        return vecinos


    def recommend(
        self,
        profile,
        config,
        top_n=10,
        exclude=()
    ):

        k = int(
            config["k"]
        )

        alpha = float(
            config["alpha"]
        )


        # -------------------------------------------------
        # 1. Calculamos el puntaje según perfiles similares
        # -------------------------------------------------

        profile_scores = Counter()

        vecinos = self.neighbors(
            profile,
            k
        )


        for uid, similitud in vecinos:

            productos_usuario = self.positives.get(
                uid,
                []
            )

            for product_id in productos_usuario:

                profile_scores[product_id] += similitud


        # Normalizamos el puntaje de perfil entre 0 y 1
        if profile_scores:

            max_profile_score = max(
                profile_scores.values()
            )

        else:

            max_profile_score = 1


        # -------------------------------------------------
        # 2. Combinamos perfil y popularidad
        # -------------------------------------------------

        productos = (
            set(profile_scores.keys())
            | set(self.popularity.keys())
        )


        resultados = []


        for product_id in productos:

            score_perfil = (
                profile_scores.get(product_id, 0)
                / max_profile_score
            )

            score_popularidad = self.popularity.get(
                product_id,
                0
            )


            # Alpha controla cuánto pesa cada señal
            score_final = (
                alpha * score_perfil
                + (1 - alpha) * score_popularidad
            )


            resultados.append({
                "product_id": product_id,
                "score": score_final
            })


        # Convertimos los resultados a DataFrame
        frame = pd.DataFrame(
            resultados
        )


        # Ordenamos los productos y devolvemos el Top-10
        return rank_scores(
            frame,
            "score",
            top_n,
            exclude
        )

## 4. Preparación del modelo y configuraciones

In [4]:
# Creamos el modelo que combina perfil y popularidad
model = ProfilePopularity(ctx)
print(
    "Usuarios de entrenamiento:",
    model.n_train_users
)
print(
    "Perfiles distintos:",
    len(model.signatures)
)
# Configuraciones que vamos a probar en validación
CONFIGURACIONES = [
    {
        "k": k,
        "alpha": alpha
    }
    for k in [100, 250, 500]
    for alpha in [0.25, 0.50, 0.75, 1.00]
    if k <= model.n_train_users
]
if not CONFIGURACIONES:
    raise ValueError(
        "No hay configuraciones válidas para evaluar"
    )

Usuarios de entrenamiento: 502216
Perfiles distintos: 1842


## 5. Ajuste en validación y evaluación final

Para evaluar el modelo se utilizan principalmente las siguientes métricas:

- **Hit Rate@10 (HR@10):** indica en cuántos usuarios el producto que ocultamos para la prueba aparece dentro de las 10 recomendaciones.

- **NDCG@10:** además de considerar si el producto aparece en el Top-10, tiene en cuenta su posición. Cuanto más arriba aparece el producto correcto, mejor es el resultado.

- **Cobertura de usuarios:** indica qué porcentaje de usuarios recibe al menos una recomendación.

- **Top-10 completo:** indica qué porcentaje de usuarios recibe las 10 recomendaciones.

- **Cobertura de catálogo:** indica qué proporción de los productos disponibles aparece al menos una vez entre las recomendaciones generadas.

En esta versión se prueban distintas combinaciones de dos parámetros:

- **K:** cantidad de usuarios vecinos considerados.
- **alpha:** peso asignado a la señal basada en perfiles frente a la popularidad.

Un valor de `alpha` cercano a 1 da mayor importancia a la similitud de perfil, mientras que un valor más bajo aumenta la influencia de la popularidad.

Las distintas combinaciones de K y alpha se evalúan primero utilizando los 500 usuarios de validación. La mejor configuración se selecciona principalmente según su HR@10 y, en caso de empate, se utiliza NDCG@10.

Una vez elegida la mejor configuración, se evalúa con los 500 usuarios de test.

Finalmente, los resultados del modelo se comparan con el mismo baseline de popularidad utilizado en los modelos anteriores.

In [5]:
mejor_config, validacion, metricas_test, detalle_test, baseline_test = sc.run_experiment(
    model,
    ctx,
    CONFIGURACIONES
)

print("Resultados de validación:")
display(validacion)

print("Configuración elegida:", mejor_config)

print("Resultados finales en test:")
display(
    pd.DataFrame(
        [metricas_test, baseline_test],
        index=["Modelo", "Baseline global"]
    )
)

Validación: {'k': 100, 'alpha': 0.25}
Evaluados 50/500
Evaluados 100/500
Evaluados 150/500
Evaluados 200/500
Evaluados 250/500
Evaluados 300/500
Evaluados 350/500
Evaluados 400/500
Evaluados 450/500
Evaluados 500/500
Validación: {'k': 100, 'alpha': 0.5}
Evaluados 50/500
Evaluados 100/500
Evaluados 150/500
Evaluados 200/500
Evaluados 250/500
Evaluados 300/500
Evaluados 350/500
Evaluados 400/500
Evaluados 450/500
Evaluados 500/500
Validación: {'k': 100, 'alpha': 0.75}
Evaluados 50/500
Evaluados 100/500
Evaluados 150/500
Evaluados 200/500
Evaluados 250/500
Evaluados 300/500
Evaluados 350/500
Evaluados 400/500
Evaluados 450/500
Evaluados 500/500
Validación: {'k': 100, 'alpha': 1.0}
Evaluados 50/500
Evaluados 100/500
Evaluados 150/500
Evaluados 200/500
Evaluados 250/500
Evaluados 300/500
Evaluados 350/500
Evaluados 400/500
Evaluados 450/500
Evaluados 500/500
Validación: {'k': 250, 'alpha': 0.25}
Evaluados 50/500
Evaluados 100/500
Evaluados 150/500
Evaluados 200/500
Evaluados 250/500
Evaluad

,config,users,hits,hit_rate_at_10,ndcg_at_10,user_coverage,full_top10_rate,catalog_coverage,mean_recommendations,mean_query_seconds
0,"{'k': 100, 'alpha': 0.25}",500,37,0.074,0.041333,1.0,1.0,0.032231,10.0,0.022385
1,"{'k': 100, 'alpha': 0.5}",500,38,0.076,0.041832,1.0,1.0,0.079339,10.0,0.022020
2,"{'k': 100, 'alpha': 0.75}",500,34,0.068,0.040903,1.0,1.0,0.104959,10.0,0.025606
3,"{'k': 100, 'alpha': 1.0}",500,27,0.054,0.033233,1.0,1.0,0.142562,10.0,0.022302
4,"{'k': 250, 'alpha': 0.25}",500,39,0.078,0.041829,1.0,1.0,0.019008,10.0,0.019238
5,"{'k': 250, 'alpha': 0.5}",500,39,0.078,0.042362,1.0,1.0,0.052893,10.0,0.019092
6,"{'k': 250, 'alpha': 0.75}",500,35,0.070,0.043784,1.0,1.0,0.080579,10.0,0.019001
7,"{'k': 250, 'alpha': 1.0}",500,34,0.068,0.043241,1.0,1.0,0.103719,10.0,0.016485
8,"{'k': 500, 'alpha': 0.25}",500,38,0.076,0.041011,1.0,1.0,0.012810,10.0,0.019794
9,"{'k': 500, 'alpha': 0.5}",500,39,0.078,0.042950,1.0,1.0,0.032645,10.0,0.018225


Configuración elegida: {'k': 500, 'alpha': 0.5}
Resultados finales en test:


,users,hits,hit_rate_at_10,ndcg_at_10,user_coverage,full_top10_rate,catalog_coverage,mean_recommendations,mean_query_seconds
Modelo,500,36,0.072,0.034156,1.0,1.0,0.033471,10.0,0.019691
Baseline global,500,37,0.074,0.033161,1.0,1.0,0.004132,10.0,0.005364


## 6. Ejemplo de recomendación para un usuario nuevo

Para visualizar cómo funcionaría el modelo en una situación real, se utiliza un perfil de ejemplo y se generan sus 10 productos recomendados.

En este caso no se está evaluando el modelo, sino mostrando el resultado que recibiría un usuario nuevo al ingresar sus características.

El modelo busca usuarios con perfiles similares y obtiene los productos con interacciones positivas de esos vecinos. Luego combina esta información con la popularidad de los productos dentro del conjunto de entrenamiento.

El peso de cada señal depende del valor de `alpha` seleccionado durante la etapa de validación.

Finalmente, los productos se ordenan según el puntaje combinado y se muestran las 10 recomendaciones con mayor puntuación.

In [6]:
# Perfil de ejemplo para generar una recomendación
PERFIL_EJEMPLO = {
    "skin_type": "oily",
    "skin_tone": "dark",
    "eye_color": "blue",
    "hair_color": "brown"
}
# Generamos las recomendaciones para este perfil
perfil, top10 = sc.example_top10(
    model,
    ctx,
    mejor_config,
    PERFIL_EJEMPLO
)
# Mostramos el perfil utilizado
display(
    pd.DataFrame([perfil])
)
# Mostramos los productos recomendados
display(top10)
# Verificamos cuántos productos fueron devueltos
print("Productos devueltos:", len(top10))

,skin_type,skin_tone,eye_color,hair_color
0,oily,dark,blue,brown


,position,product_id,product_name,brand_name,secondary_category,price_usd
0,1,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,Lip Balms & Treatments,24.0
1,2,P427417,Niacinamide 10% + Zinc 1% Oil Control Serum,The Ordinary,Treatments,6.0
2,3,P411540,Cicapair Tiger Grass Color Correcting Treatmen...,Dr. Jart+,Moisturizers,52.0
3,4,P7880,Soy Hydrating Gentle Face Cleanser,fresh,Cleansers,39.0
4,5,P471546,Mini Cicapair Tiger Grass Color Correcting Tre...,Dr. Jart+,Mini Size,21.0
5,6,P427421,Protini Polypeptide Firming Refillable Moistur...,Drunk Elephant,Moisturizers,68.0
6,7,P423688,Daily Microfoliant Exfoliator,Dermalogica,Cleansers,65.0
7,8,P443352,Mini Daily Microfoliant Exfoliator,Dermalogica,Mini Size,18.0
8,9,P417238,Green Clean Makeup Removing Cleansing Balm,Farmacy,Cleansers,36.0
9,10,P450271,Green Clean Makeup Meltaway Cleansing Balm Lim...,Farmacy,Cleansers,60.0


Productos devueltos: 10


## 7. Cierre y resultados

El Modelo A2 incorporó la popularidad de los productos a la señal obtenida a partir de usuarios con perfiles similares.

Durante la etapa de validación, la configuración seleccionada fue:

- K = 500 vecinos.
- alpha = 0.50.

Esto significa que el puntaje final combina en partes iguales la señal basada en perfiles similares y la popularidad de los productos.

En el conjunto de test, el modelo obtuvo:

- HR@10 = 7,2 %, con 36 aciertos sobre 500 usuarios.
- NDCG@10 = 0,0342.
- Cobertura de usuarios = 100 %.
- Top-10 completo = 100 %.

El baseline de popularidad obtuvo un HR@10 de 7,4 %, con 37 aciertos, y un NDCG@10 de 0,0332.

Por lo tanto, A2 quedó a un solo acierto del baseline en HR@10, pero logró superarlo en NDCG@10. Esto indica que, aunque el baseline encontró el producto objetivo para un usuario más, A2 logró ubicar mejor los productos relevantes dentro del ranking.

En comparación con las versiones anteriores, A2 muestra una mejora importante y sugiere que combinar la información del perfil con una señal de popularidad puede generar recomendaciones más competitivas.

In [8]:
# Mostramos la conclusión del modelo comparada con el baseline
sc.conclusion(
    metricas_test,
    baseline_test
)
# Información adicional que identifica el algoritmo utilizado
extra = {
    "algorithm": "ProfilePopularity",
    "version": "perfil_popularidad"
}
# Guardamos los resultados del experimento
archivo = sc.save_result(
    ctx,
    "01_A2_perfil_popularidad",
    mejor_config,
    validacion,
    metricas_test,
    detalle_test,
    baseline_test,
    top10,
    extra
)
print(
    "Resultado guardado:",
    archivo
)

HR@10 test: 7.20%; baseline: 7.40%; diferencia: -0.20 puntos porcentuales.
Cobertura de usuarios: 100.00%; catálogo: 3.35%.
En esta muestra no supera al baseline en HR. No declararlo ganador por personalización solamente.
Un hit recupera una preferencia observada; no prueba eficacia ni compatibilidad dermatológica.
Resultado guardado: c:\Users\Administrator\Documents\DATA SCIENCE\PF\sephora-recommender\artifacts\comparacion_4_modelos\99fc64b0ea63\01_A2_perfil_popularidad.json
